# Part 4: Indexes and Query Performance

Indexes make queries faster by allowing SQLite to find rows without scanning the entire table. In this notebook we:

- Inspect which indexes already exist (primary keys are indexed automatically)
- Create new indexes on frequently queried columns
- Use `EXPLAIN QUERY PLAN` to see how SQLite executes a query
- Observe the difference in plans before and after indexing

> **Prerequisite:** Run `01_setup.ipynb` first.

In [ ]:
import sqlite3
import pandas as pd
import time

con = sqlite3.connect("gene_expression.db")

def query(sql, params=()):
    return pd.read_sql(sql, con, params=params)

print("Connected.")

---
## 1. List existing indexes

In [ ]:
query("""
SELECT name, tbl_name, sql
FROM   sqlite_master
WHERE  type = 'index'
ORDER BY tbl_name, name
""")

SQLite automatically creates indexes for `PRIMARY KEY` and `UNIQUE` constraints. The `expression` table has a unique constraint on `(gene_id, sample_id)` which is already indexed.

---
## 2. EXPLAIN QUERY PLAN — before adding an index

`EXPLAIN QUERY PLAN` shows which strategy SQLite will use to execute a query, without actually running it.

In [ ]:
# Query: find all expression rows for a specific gene symbol
# This requires looking up the gene by symbol first, then joining expression.

sql = """
SELECT g.symbol, s.name, e.tpm
FROM   expression e
JOIN   genes      g ON g.gene_id   = e.gene_id
JOIN   samples    s ON s.sample_id = e.sample_id
WHERE  g.symbol = 'BRCA1'
"""

query("EXPLAIN QUERY PLAN " + sql)

Look for **SCAN** vs **SEARCH** in the detail column:
- `SCAN` — reads every row (slow for large tables)
- `SEARCH` — uses an index to jump directly to matching rows (fast)

---
## 3. Create indexes on commonly filtered columns

In [ ]:
# Index on genes.chromosome — for queries that filter by chromosome
con.execute("CREATE INDEX IF NOT EXISTS idx_genes_chromosome ON genes(chromosome)")

# Index on samples.condition — for filtering tumor vs normal
con.execute("CREATE INDEX IF NOT EXISTS idx_samples_condition ON samples(condition)")

# Index on expression.tpm — for filtering by expression level
con.execute("CREATE INDEX IF NOT EXISTS idx_expression_tpm ON expression(tpm)")

con.commit()
print("Indexes created.")

---
## 4. EXPLAIN QUERY PLAN — after adding indexes

In [ ]:
# Same query — does SQLite now use an index?
query("EXPLAIN QUERY PLAN " + sql)

In [ ]:
# Query that filters on chromosome — should now use idx_genes_chromosome
query("""
EXPLAIN QUERY PLAN
SELECT symbol, chromosome
FROM   genes
WHERE  chromosome = '17'
""")

In [ ]:
# Query filtering on tpm — should now use idx_expression_tpm
query("""
EXPLAIN QUERY PLAN
SELECT gene_id, tpm
FROM   expression
WHERE  tpm > 1500
""")

---
## 5. Timing queries (optional — with a larger synthetic dataset)

Our dataset is small (150 rows), so timing differences are negligible. The cell below inserts a large number of synthetic rows so you can observe a real timing difference.

In [ ]:
import random
random.seed(99)

# Drop the tpm index temporarily
con.execute("DROP INDEX IF EXISTS idx_expression_tpm")
con.commit()

# Insert 50,000 synthetic expression rows (fake gene/sample ids within range)
gene_ids   = [r[0] for r in con.execute("SELECT gene_id FROM genes")]
sample_ids = [r[0] for r in con.execute("SELECT sample_id FROM samples")]

# We will use a temporary in-memory table to avoid polluting the real data
con2 = sqlite3.connect(":memory:")
con2.executescript("""
CREATE TABLE big_expression (
    id  INTEGER PRIMARY KEY,
    tpm REAL NOT NULL
);
""")

big_rows = [(round(random.uniform(0, 5000), 2),) for _ in range(500_000)]
con2.executemany("INSERT INTO big_expression (tpm) VALUES (?)", big_rows)
con2.commit()
print(f"Inserted {len(big_rows):,} rows into in-memory table.")

# Time query WITHOUT index
t0 = time.perf_counter()
result = con2.execute("SELECT COUNT(*) FROM big_expression WHERE tpm > 4500").fetchone()
t1 = time.perf_counter()
print(f"\nWithout index: {result[0]:,} rows matched, took {(t1-t0)*1000:.1f} ms")

# Create index
con2.execute("CREATE INDEX idx_tpm ON big_expression(tpm)")
con2.commit()

# Time query WITH index
t0 = time.perf_counter()
result = con2.execute("SELECT COUNT(*) FROM big_expression WHERE tpm > 4500").fetchone()
t1 = time.perf_counter()
print(f"With index:    {result[0]:,} rows matched, took {(t1-t0)*1000:.1f} ms")

con2.close()

---
## 6. Drop an index

Indexes cost disk space and slow down `INSERT`/`UPDATE`. Drop them when they are not needed.

In [ ]:
# Demonstrate DROP INDEX (we will recreate it)
con.execute("DROP INDEX IF EXISTS idx_expression_tpm")
con.commit()
print("Index dropped.")

# Recreate
con.execute("CREATE INDEX idx_expression_tpm ON expression(tpm)")
con.commit()
print("Index recreated.")

---
## Summary

| Concept | Key takeaway |
|---|---|
| `EXPLAIN QUERY PLAN` | Shows whether SQLite does a SCAN or SEARCH |
| `CREATE INDEX` | Speeds up `WHERE`, `JOIN`, `ORDER BY` on indexed columns |
| `DROP INDEX` | Removes an index to save space or improve write speed |
| Trade-off | Indexes speed up reads but slow down writes and use disk space |

In [ ]:
con.close()